# 05. Temporal modelling: predict student outcome from pre-last-attempt history

## Critical target interpretation
The source contains a **student-level dropout label but no dropout timestamp**. We cannot truthfully claim that dropout happened at the last attempt. This notebook therefore treats the last observed attempt as a **prediction landmark** and predicts the student's supplied outcome label from history strictly before that last attempt. The label means “eventually labelled dropout in this dataset,” not “dropout occurred at this time.”

## Design
- Build one candidate row for each attempt that is **before that student's final observed attempt**. Features on each row are strictly past-only (from `temporal_attempt_sequences.csv`).
- Split by **student**, never by attempt, so all rows from a student remain in one partition.
- Train on attempt-level rows, but evaluate student-wise by selecting each student's latest eligible prediction row. This prevents prolific students from dominating the reported metrics.
- Report two evaluation regimes:
  1. Random student split = interpolation.
  2. Temporal student split using each student's last observed attempt date = extrapolation to later observed activity periods. Entire students are held out; no student's rows cross splits.
- Temporal split is a proxy only: because dropout time is missing, it is not a true time-to-dropout evaluation.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix
from IPython.display import display

ROOT = Path("..").resolve()
DATA = ROOT / "data" / "curated"
OUT = ROOT / "outputs" / "metrics"
OUT.mkdir(parents=True, exist_ok=True)
seq = pd.read_csv(DATA / "temporal_attempt_sequences.csv", parse_dates=["attempt_dt", "signup_dt"])


In [2]:
# Sort chronologically, retain only pre-final-attempt rows for each student.
seq = seq.sort_values(["student_id", "attempt_dt", "attempt_id"]).copy()
last_idx = seq.groupby("student_id").tail(1).index
samples = seq.loc[~seq.index.isin(last_idx)].copy()
samples["student_last_attempt_dt"] = seq.groupby("student_id")["attempt_dt"].transform("max").reindex(samples.index)
# Exclude the current attempt's score/duration/lesson: prediction is immediately before that attempt.
# Prior-history features are the only behavioural features used.
feature_cols = [
    "days_from_signup", "sequence_index", "gap_since_previous_attempt_days",
    "prior_valid_score_count", "prior_score_mean_3", "prior_score_std_3",
    "prior_score_mean_all", "prior_lesson_count", "duration_missing_flag",
    "duration_negative_flag"
]
# Include baseline signup-time demographics, but not IDs, raw timestamps, target semantics, or current attempt outcomes.
for c in ["age_group", "country", "device"]:
    if c in seq.columns: feature_cols.append(c)
samples = samples.dropna(subset=["target_dropout"])
assert (samples.groupby("student_id").size() >= 1).all()
assert (samples.groupby("student_id")["sequence_index"].max() < seq.groupby("student_id")["sequence_index"].max().reindex(samples.student_id.unique()).values).all()
print("Candidate pre-final-attempt rows:", len(samples), "| students:", samples.student_id.nunique())
print("Students with only one attempt excluded:", seq.student_id.nunique() - samples.student_id.nunique())


Candidate pre-final-attempt rows: 1954 | students: 478
Students with only one attempt excluded: 21


## Feature safety check

The current attempt's score and duration are deliberately not used: this is a prediction made before that attempt. `prior_*` features were computed using only earlier attempts. Student-level demographic variables are included as baseline covariates. We exclude `refund_requested` because it can leak the outcome.

In [3]:
def make_pipeline(df):
    cat = [c for c in feature_cols if c in df.columns and (df[c].dtype == "object" or str(df[c].dtype).startswith("category"))]
    num = [c for c in feature_cols if c in df.columns and c not in cat]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat)
    ])
    return Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42))])

def metrics(y, p, threshold=0.5):
    pred = p >= threshold
    cm = confusion_matrix(y, pred, labels=[0,1])
    try: pr = average_precision_score(y,p)
    except ValueError: pr = np.nan
    try: roc = roc_auc_score(y,p)
    except ValueError: roc = np.nan
    return {"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),
            "f1":f1_score(y,pred,zero_division=0),"PR_AUC":pr,"ROC_AUC":roc,
            "TN":cm[0,0],"FP":cm[0,1],"FN":cm[1,0],"TP":cm[1,1]}


## Regime A: random student-level split (interpolation)

Rows are used for fitting, but the final metric is computed once per student using their latest eligible pre-final-attempt row. The validation set chooses a threshold for F1; the test set is untouched until reporting.

In [4]:
def run_random_student_split(samples, seed=42):
    ids = samples[["student_id","target_dropout"]].drop_duplicates("student_id")
    tr_ids, tmp_ids = train_test_split(ids.student_id, test_size=0.40, random_state=seed,
        stratify=ids.set_index("student_id").loc[ids.student_id,"target_dropout"])
    tmp = ids[ids.student_id.isin(tmp_ids)]
    va_ids, te_ids = train_test_split(tmp.student_id, test_size=0.50, random_state=seed,
        stratify=tmp.target_dropout)
    tr = samples[samples.student_id.isin(tr_ids)]
    va = samples[samples.student_id.isin(va_ids)]
    te = samples[samples.student_id.isin(te_ids)]
    model = make_pipeline(samples)
    model.fit(tr[feature_cols], tr.target_dropout.astype(int))
    # Student-wise: latest eligible pre-final-attempt prediction per student
    va_one = va.sort_values("attempt_dt").groupby("student_id").tail(1)
    pv = model.predict_proba(va_one[feature_cols])[:,1]
    thresholds = np.arange(0.10,0.91,0.025)
    threshold = max(thresholds, key=lambda t:(f1_score(va_one.target_dropout.astype(int),pv>=t,zero_division=0),t))
    te_one = te.sort_values("attempt_dt").groupby("student_id").tail(1)
    pt = model.predict_proba(te_one[feature_cols])[:,1]
    result = metrics(te_one.target_dropout.astype(int).to_numpy(),pt,threshold)
    result.update({"regime":"random_student_interpolation","threshold":threshold,
                   "train_students":tr.student_id.nunique(),"validation_students":va_one.student_id.nunique(),
                   "test_students":te_one.student_id.nunique(),"train_attempt_rows":len(tr)})
    return result, model, te_one.assign(predicted_probability=pt, predicted_dropout=(pt>=threshold))
random_result, random_model, random_student_predictions = run_random_student_split(samples)
display(pd.DataFrame([random_result]).round(3))
random_student_predictions.to_csv(OUT / "temporal_random_student_predictions.csv", index=False)


,precision,recall,f1,PR_AUC,ROC_AUC,TN,FP,FN,TP,regime,threshold,train_students,validation_students,test_students,train_attempt_rows
0,0.536,0.714,0.612,0.649,0.866,62,13,6,15,random_student_interpolation,0.525,286,96,96,1192


## Regime B: temporal student-level split (extrapolation proxy)

Students are ordered by their **last observed attempt timestamp**. Earlier 60% of students form training, next 20% validation, latest 20% test. All attempts belonging to a student stay together. This answers whether the model transfers to students whose observed activity ends later in calendar time. It is not a genuine future-dropout forecast because outcome timestamps are unavailable.

In [5]:
def run_temporal_student_split(samples):
    student_dates = samples.groupby("student_id").agg(
        last_landmark=("student_last_attempt_dt","max"),
        target_dropout=("target_dropout","first")
    ).reset_index().sort_values(["last_landmark","student_id"])
    # Keep chronological ordering; boundaries are by student, not by attempt row.
    n = len(student_dates)
    tr_ids = set(student_dates.iloc[:int(0.60*n)].student_id)
    va_ids = set(student_dates.iloc[int(0.60*n):int(0.80*n)].student_id)
    te_ids = set(student_dates.iloc[int(0.80*n):].student_id)
    assert not (tr_ids & va_ids or tr_ids & te_ids or va_ids & te_ids)
    tr, va, te = [samples[samples.student_id.isin(s)].copy() for s in (tr_ids,va_ids,te_ids)]
    model = make_pipeline(samples)
    model.fit(tr[feature_cols], tr.target_dropout.astype(int))
    va_one = va.sort_values("attempt_dt").groupby("student_id").tail(1)
    pv = model.predict_proba(va_one[feature_cols])[:,1]
    thresholds = np.arange(0.10,0.91,0.025)
    threshold = max(thresholds, key=lambda t:(f1_score(va_one.target_dropout.astype(int),pv>=t,zero_division=0),t))
    te_one = te.sort_values("attempt_dt").groupby("student_id").tail(1)
    pt = model.predict_proba(te_one[feature_cols])[:,1]
    result = metrics(te_one.target_dropout.astype(int).to_numpy(),pt,threshold)
    result.update({"regime":"temporal_student_extrapolation_proxy","threshold":threshold,
                   "train_students":tr.student_id.nunique(),"validation_students":va_one.student_id.nunique(),
                   "test_students":te_one.student_id.nunique(),"train_attempt_rows":len(tr),
                   "train_last_date":str(student_dates.iloc[int(0.60*n)-1].last_landmark.date()),
                   "validation_last_date":str(student_dates.iloc[int(0.80*n)-1].last_landmark.date()),
                   "test_first_date":str(student_dates.iloc[int(0.80*n)].last_landmark.date())})
    return result, model, te_one.assign(predicted_probability=pt, predicted_dropout=(pt>=threshold))
temporal_result, temporal_model, temporal_student_predictions = run_temporal_student_split(samples)
display(pd.DataFrame([temporal_result]).round(3))
temporal_student_predictions.to_csv(OUT / "temporal_extrapolation_student_predictions.csv", index=False)
pd.DataFrame([random_result, temporal_result]).to_csv(OUT / "temporal_model_regime_comparison.csv", index=False)


,precision,recall,f1,PR_AUC,ROC_AUC,TN,FP,FN,TP,regime,threshold,train_students,validation_students,test_students,train_attempt_rows,train_last_date,validation_last_date,test_first_date
0,0.214,1.0,0.353,0.472,0.961,82,11,0,3,temporal_student_extrapolation_proxy,0.1,286,96,96,1047,2026-08-20,2026-09-01,2026-09-01


## Reading the results

- **Student-wise scoring** means every student contributes exactly one prediction to precision/recall/F1/PR-AUC/ROC-AUC, even though model fitting uses multiple attempt rows per student.
- **Never split attempts randomly.** That would put the same student's patterns in train and test and inflate performance.
- Random student split is an interpolation estimate. The chronological student split is a stricter extrapolation proxy, but it sorts by each student's last observed attempt, so it is a retrospective evaluation design.
- The latest eligible row may be very close to the final attempt and uses all prior attempts available at that point. It does not simulate a fixed number-of-days intervention. For a deployable early-warning system, define an operational prediction schedule and obtain the true dropout timestamp or a consistent label window.
- Students with one attempt cannot provide a pre-final-attempt row and are excluded from this temporal experiment. Report this limitation alongside results.
